# Train biped_rl on Kaggle (clone from GitHub)
1. Push this repo to GitHub first (the branch below must contain `scripts/` and `biped_rl/`).
2. Notebook settings: **Internet on**. Accelerator: none (physics runs on CPU; ~30 GB RAM either way).
3. Private repo: Add-ons -> Secrets -> add `GITHUB_TOKEN` (a read-only personal access token) and tick it for this notebook.
4. To continue a run, optionally add a Kaggle Dataset holding that run's newest `ckpt_<N>_steps.zip` + `ckpt_vecnormalize_<N>_steps.pkl` (any folder layout). Without it, training starts from scratch.
5. *Save Version -> Save & Run All*. It stops itself at `MAX_HOURS` (Kaggle kills at 12 h). Download `<RUN>_out.zip` from Output.

In [ ]:
REPO_URL = "https://github.com/MilesDao/biped_rl.git"   # edit: your repo
BRANCH = "stand-then-walk"
RUN, START_LEVEL, KP_SCALE, STEPS = "stand_b", 3, 3, 3_000_000
MAX_HOURS = 11   # Kaggle hard limit is 12 h; leave 1 h for saving + zipping

In [ ]:
import glob, os, re, shutil, subprocess
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    pass  # public repo, no token needed
os.chdir("/kaggle/working")
if not os.path.isdir("biped_rl/.git"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "biped_rl"], check=True)
os.chdir("/kaggle/working/biped_rl")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout, "cpus:", os.cpu_count())

In [ ]:
!pip install -q "mujoco>=3.2" "gymnasium>=1.0" "stable-baselines3>=2.3" imageio-ffmpeg
!pip install -q -e . --no-deps

In [ ]:
# continue from the newest checkpoint found in any attached dataset, else start fresh
run = f"runs/{RUN}"
os.makedirs(run, exist_ok=True)
ckpts = {int(re.findall(r"ckpt_(\d+)_steps", f)[0]): f for f in glob.glob("/kaggle/input/**/ckpt_*_steps.zip", recursive=True)}
RESUME = ""
if ckpts:
    steps = max(ckpts)
    stats = os.path.join(os.path.dirname(ckpts[steps]), f"ckpt_vecnormalize_{steps}_steps.pkl")
    shutil.copy(ckpts[steps], f"{run}/latest.zip")
    shutil.copy(stats, f"{run}/vecnormalize.pkl")
    RESUME = f"--resume {run}/latest.zip"
    print("resuming from step", steps)
else:
    START_LEVEL = 0
    print("no checkpoint attached: training from scratch at level 0")

In [ ]:
N_ENVS = os.cpu_count()
# stop at MAX_HOURS with Ctrl-C so train.py still saves latest.zip
!PYTHONUNBUFFERED=1 timeout --signal=INT {MAX_HOURS}h python scripts/train.py --name {RUN} {RESUME} --start-level {START_LEVEL}     --kp-scale {KP_SCALE} --total-steps {STEPS} --n-envs {N_ENVS} --save-every 250000 2>&1 | tee -a runs/{RUN}/train.log | grep -E "level|speed |ep_len_mean|fps|total_timesteps|Error" 

In [ ]:
shutil.make_archive(f"/kaggle/working/{RUN}_out", "zip", "runs", RUN)
print("download", f"{RUN}_out.zip", "from the Output tab, unzip into runs/")